# The first big project - The Digital Twin

## But first: introducing Pushover

**Pushover** is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

1. Simply visit [https://pushover.net/](https://pushover.net/) and click **'Login or Signup'** on the top right to sign up for a free account, and create your API keys.
2. Once you've signed up, on the home screen, click **"Create an Application/API Token"**, give it any name (like `Agents`), and click **Create Application**.
3. Then add 2 lines to your `.env` file:

In [1]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
# The usual start

load_dotenv(override=True)
openai = OpenAI()

In [3]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


#### <span style="color:#1E88E5">Send Push Notifications via Pushover</span>

This function sends a message to your device through the Pushover API, using the configured user key and app token.
It reports whether the notification was delivered, and warns you if the Pushover credentials haven't been set.

In [4]:
def push(message):
    if pushover_user and pushover_token:
        data = {
            "token": pushover_token,
            "user": pushover_user,
            "message": message
        }
        response = requests.post(pushover_url, data=data)
        if response.status_code == 200:
            print("Pushover notification sent successfully.")
        else:
            print(f"Failed to send pushover notification. Status code: {response.status_code}")
    else:
        print("Pushover credentials not set. Cannot send notification.")

In [6]:
push("hey")

Pushover notification sent successfully.


In [8]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"

In [9]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [10]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [11]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [12]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [13]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional info about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['question'],


In [14]:
def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        if tool_call.name == "record_user_details":
            email = json.loads(tool_call.arguments).get("email")
            name = json.loads(tool_call.arguments).get("name", "Name not provided")
            notes = json.loads(tool_call.arguments).get("notes", "not provided")
            result = record_user_details(email, name, notes)
        elif tool_call.name == "record_unknown_question":
            question = json.loads(tool_call.arguments).get("question")
            result = record_unknown_question(question)
        else:
            print(f"Unknown tool call: {tool_call.name}")

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results


In [15]:
globals()["record_unknown_question"]("this is a really hard question")

Pushover notification sent successfully.


'OK'

In [16]:
# This gives us a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [17]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [18]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""

In [21]:
def chat(message, history):

    # Convert Gradio history format to OpenAI message format
    # Gradio history is a list of [user_message, assistant_response] pairs
    messages = [{"role": "system", "content": system_prompt}]
    
    for h in history:
        if isinstance(h, (list, tuple)) and len(h) == 2:
            # Gradio format: [user_message, assistant_response]
            messages.append({"role": "user", "content": h[0]})
            messages.append({"role": "assistant", "content": h[1]})
        elif isinstance(h, dict):
            # Already in dict format
            messages.append(h)
    
    # Add current user message
    messages.append({"role": "user", "content": message})
    
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
 
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

/Users/bhaveshg/Study/bg-ai-hub/ai-agents/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


Tool called: record_unknown_question
Pushover notification sent successfully.
Tool called: record_user_details
Pushover notification sent successfully.
Tool called: record_user_details
Pushover notification sent successfully.
